# Lab 04 (Sample) — Cost Function, Explained Simply

This notebook covers the **same idea** as the original Lab 04, with more explanation.

## Big picture (read this first)

Think of learning as 3 steps:

| Step | What it is | Simple meaning |
|------|------------|----------------|
| 1. Model | $f_{w,b}(x) = wx + b$ | How we make a prediction |
| 2. Parameters | $w$, $b$ | The dials we can turn |
| 3. Cost $J(w,b)$ | a number | How wrong the predictions are |

**Goal of this lab:** understand the cost function, then understand what **minimize $J$** means.

> In Lab 03 we used $w$ and $b$ by hand.  
> In Lab 04 we learn how to **score** a choice of $w$ and $b`.  
> Later (gradient descent) we learn how to **search** for the best $w$ and $b$ automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    plt.style.use('./deeplearning.mplstyle')
except OSError:
    pass

## 1. Quick recap — model and parameters

### Model
The model is the prediction formula:

$$
f_{w,b}(x) = wx + b
$$

Example: if $w=200$, $b=100$, and house size $x=1.0$:

$$
f = 200 \times 1.0 + 100 = 300
$$

### Parameters ($w$, $b$)
Parameters are the values inside the model that we can change:

- $w$ = slope ("price change per unit size")
- $b$ = intercept ("base price")

Different $(w,b)$ → different prediction line → different quality of fit.

### Important
- The **model formula stays the same**: $wx + b$
- What we change is the **parameter values** $w$ and $b$

## 2. Training data (same tiny example)

| House | Size $x$ (1000 sqft) | Price $y$ (1000s of dollars) |
|-------|----------------------|------------------------------|
| 1 | 1.0 | 300 |
| 2 | 2.0 | 500 |

We already know the true prices $y$.  
The model will guess prices $\hat{y} = f_{w,b}(x)$.  
Cost measures: **how far are the guesses from the truth?**

In [ ]:
x_train = np.array([1.0, 2.0])    # size
y_train = np.array([300.0, 500.0])  # true price

print(f"x_train = {x_train}")
print(f"y_train = {y_train}")
print(f"m (number of examples) = {len(x_train)}")

## 3. What is the cost function?

### Plain English

**Cost** = a score of how bad the current $w$ and $b$ are.

- Low cost → predictions are close to real $y$ → good parameters
- High cost → predictions are far from real $y$ → bad parameters

(Here "cost" does **not** mean house price. House price is $y$. Cost is the error score $J$.)

### Formula

$$
J(w,b) = \frac{1}{2m} \sum_{i=0}^{m-1} \left( f_{w,b}(x^{(i)}) - y^{(i)} \right)^2
$$

with

$$
f_{w,b}(x^{(i)}) = w x^{(i)} + b
$$

### Read it piece by piece

| Piece | Meaning |
|-------|---------|
| $f_{w,b}(x^{(i)})$ | prediction for example $i$ |
| $y^{(i)}$ | true target for example $i$ |
| $f - y$ | error (how wrong we are) |
| $(f - y)^2$ | squared error (positive, punishes big mistakes more) |
| $\sum$ | add error from all examples |
| $\frac{1}{2m}$ | average-style scaling (the $2$ makes later math nicer) |

### One-line mental model

> $J(w,b)$ = "average squared prediction error for this choice of $w$ and $b$."

## 4. Compute cost by hand (best way to understand)

Let's try **bad** parameters first:

- $w = 100$
- $b = 100$

### House 0 ($x=1$, $y=300$)

$$
f = 100 \times 1 + 100 = 200
$$

$$
(f - y)^2 = (200 - 300)^2 = (-100)^2 = 10000
$$

### House 1 ($x=2$, $y=500$)

$$
f = 100 \times 2 + 100 = 300
$$

$$
(f - y)^2 = (300 - 500)^2 = (-200)^2 = 40000
$$

### Total cost

$$
J = \frac{1}{2 \times 2} (10000 + 40000) = \frac{1}{4} \times 50000 = 12500
$$

So with $w=100$, $b=100$, cost is **12500** (quite high → bad fit).

In [ ]:
w_bad = 100
b_bad = 100

f0 = w_bad * x_train[0] + b_bad
f1 = w_bad * x_train[1] + b_bad

err0 = (f0 - y_train[0]) ** 2
err1 = (f1 - y_train[1]) ** 2
J_bad = (1 / (2 * 2)) * (err0 + err1)

print(f"predictions: {[f0, f1]}")
print(f"actual y:    {list(y_train)}")
print(f"squared errors: {[err0, err1]}")
print(f"J(w={w_bad}, b={b_bad}) = {J_bad}")

## 5. Now try good parameters

From Lab 03, a good fit was:

- $w = 200$
- $b = 100$

### House 0

$$
f = 200 \times 1 + 100 = 300, \quad (300-300)^2 = 0
$$

### House 1

$$
f = 200 \times 2 + 100 = 500, \quad (500-500)^2 = 0
$$

### Total cost

$$
J = \frac{1}{4}(0 + 0) = 0
$$

Cost **0** means perfect predictions on this tiny dataset.

> Perfect zero cost is possible here because both points lie on one straight line.  
> With messy real data, minimum cost is usually **small but not zero**.

In [ ]:
w_good = 200
b_good = 100

f0 = w_good * x_train[0] + b_good
f1 = w_good * x_train[1] + b_good

err0 = (f0 - y_train[0]) ** 2
err1 = (f1 - y_train[1]) ** 2
J_good = (1 / (2 * 2)) * (err0 + err1)

print(f"predictions: {[f0, f1]}")
print(f"actual y:    {list(y_train)}")
print(f"J(w={w_good}, b={b_good}) = {J_good}")

## 6. `compute_cost` — same math in a function

This function does exactly what we did by hand:

1. for each example, predict $f = wx + b$
2. square the error $(f - y)^2$
3. add all squared errors
4. divide by $2m$

In [ ]:
def compute_cost(x, y, w, b):
    """
    Score how bad parameters (w, b) are on this dataset.

    Lower J = better fit.
    """
    m = x.shape[0]
    cost_sum = 0.0

    for i in range(m):
        f_wb = w * x[i] + b          # model prediction
        error = (f_wb - y[i]) ** 2   # squared error
        cost_sum = cost_sum + error

    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost


print('bad params  J =', compute_cost(x_train, y_train, w=100, b=100))
print('good params J =', compute_cost(x_train, y_train, w=200, b=100))

## 7. What does "minimize $J$" mean?

### Simple meaning

**Minimize $J(w,b)$** means:

> Find values of $w$ and $b$ that make the cost as **small as possible**.

Why? Because small cost ≈ predictions match the training data well.

### Visual idea

- Try many $w$ values (keep $b=100$ fixed for simplicity)
- Compute $J$ for each
- The $w$ with the **lowest** $J$ is the best among those tries

That lowest point is the minimum of the cost curve.

In [ ]:
b_fixed = 100
w_values = np.arange(0, 401, 50)  # try w = 0, 50, 100, ..., 400
costs = []

for w in w_values:
    costs.append(compute_cost(x_train, y_train, w, b_fixed))

for w, J in zip(w_values, costs):
    print(f"w = {w:3d}  ->  J = {J:8.1f}")

best_index = int(np.argmin(costs))
print()
print(f"Best among these tries: w = {w_values[best_index]}, J = {costs[best_index]}")

In [ ]:
plt.plot(w_values, costs, marker='o')
plt.xlabel('w (with b fixed at 100)')
plt.ylabel('Cost J(w, b)')
plt.title('Cost vs w — we want the lowest point')
plt.grid(True, alpha=0.3)
plt.show()

You should see the lowest cost near **$w = 200$**.  
That is what "minimize $J$" looks like in a graph: go to the bottom of the curve.

## 8. Connect cost back to the prediction line

Let's compare a bad line and a good line side by side with their costs.

In [ ]:
def predict(x, w, b):
    return w * x + b


fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bad model
w1, b1 = 100, 100
pred1 = predict(x_train, w1, b1)
axes[0].scatter(x_train, y_train, marker='x', c='r', label='Actual y')
axes[0].plot(x_train, pred1, c='b', label='Prediction')
axes[0].set_title(f'Bad fit\nw={w1}, b={b1}, J={compute_cost(x_train, y_train, w1, b1):.0f}')
axes[0].set_xlabel('Size (1000 sqft)')
axes[0].set_ylabel('Price (1000s of dollars)')
axes[0].legend()

# Good model
w2, b2 = 200, 100
pred2 = predict(x_train, w2, b2)
axes[1].scatter(x_train, y_train, marker='x', c='r', label='Actual y')
axes[1].plot(x_train, pred2, c='b', label='Prediction')
axes[1].set_title(f'Good fit\nw={w2}, b={b2}, J={compute_cost(x_train, y_train, w2, b2):.0f}')
axes[1].set_xlabel('Size (1000 sqft)')
axes[1].set_ylabel('Price (1000s of dollars)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Common confusion checklist

| Question | Clear answer |
|----------|--------------|
| What is the model? | $f_{w,b}(x)=wx+b$ (how we predict) |
| What are parameters? | $w$ and $b$ (values we choose / learn) |
| What is cost $J$? | A number that says how wrong those $w,b$ are |
| Does cost find $w,b$ by itself? | **No.** Cost only **scores** them |
| What does minimize $J$ mean? | Search for $w,b$ that make this score as small as possible |
| Why plot graphs? | To **see** fit quality and where cost is low — not the main algorithm for finding $w,b$ |
| What comes next? | Gradient descent: an automatic way to minimize $J$ |

## 10. Optional: more points (cost may not reach 0)

If data is not perfectly on one line, even the best $w,b$ will have some leftover error.

In [ ]:
x2 = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y2 = np.array([250.0, 300.0, 480.0, 430.0, 630.0, 730.0])

# A reasonably good guess from the original lab discussion
w_try, b_try = 209, 2.4
J_try = compute_cost(x2, y2, w_try, b_try)

print(f"J(w={w_try}, b={b_try}) on larger data = {J_try:.2f}")
print('Notice: minimum cost is small, but usually not exactly 0.')

pred = w_try * x2 + b_try
plt.scatter(x2, y2, marker='x', c='r', label='Actual y')
plt.plot(x2, pred, c='b', label='Prediction')
plt.title(f'More points — best line still has some error\nJ ≈ {J_try:.1f}')
plt.xlabel('Size (1000 sqft)')
plt.ylabel('Price (1000s of dollars)')
plt.legend()
plt.show()

## Quick recap

1. **Model:** $f_{w,b}(x) = wx + b$ makes predictions.
2. **Parameters:** $w$ and $b` control that model.
3. **Cost $J(w,b)$:** scores how wrong those parameters are (using squared errors).
4. **Minimize $J$:** choose / learn $w$ and $b` so this score is as small as possible.
5. Graphs help you **understand**; the cost formula is the actual measuring tool.

### One sentence to remember

> The model predicts with $wx+b$; the cost function grades that prediction; minimizing cost means finding the best grade (lowest $J$).